# Main Figure 5

Frozen drawing logic and saved derived inputs. Outputs: `outputs/main/figure_05`. See reproduction status before release.


In [ ]:
from pathlib import Path
import os, sys
sys.dont_write_bytecode = True
_start = Path.cwd().resolve()
REPO = next((p for q in [_start, *_start.parents] for p in [q, q / "github"]
             if (p / "plot_utils" / "release_runtime.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Run from the repository root or a notebook subdirectory.")
sys.path.insert(0, str(REPO / "plot_utils"))
from release_runtime import configure
configure(REPO)


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, tempfile
sys.dont_write_bytecode=True
PROJECT_ROOT=ROOT=REPO/'figure_source_data/shared'
OUTPUT_ROOT=REPO/'outputs/main/figure_05'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RUNTIME=OUTPUT_ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
FIG=ROOT/'fig'
for key in ['MPLCONFIGDIR','XDG_CACHE_HOME','TEMP','TMP','TMPDIR','NUMBA_CACHE_DIR']:
    os.environ[key]=str(RUNTIME)
tempfile.tempdir=str(RUNTIME)

PART7_ROOT=ROOT/'external/part7'


In [ ]:

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.ticker import ScalarFormatter
from scipy.ndimage import gaussian_filter1d
import cartopy
import cartopy.crs as ccrs
from cartopy.io import shapereader
cartopy.config['cache_dir'] = RUNTIME / 'cartopy'
import geopandas as gpd



plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Liberation Sans']
plt.rcParams['svg.fonttype']='none'


In [ ]:

SOURCE=ROOT/'Part8_change_work/figure_source_data'
TABLES=PART7_ROOT/'tables'
hashes={}
def get(path):
    path=Path(path);hashes[str(path)]=hashlib.sha256(path.read_bytes()).hexdigest()
    return pd.read_csv(path)
deciles=get(SOURCE/'Figure5_source_data_deciles.csv')
interaction=get(SOURCE/'Figure5_source_data_interaction.csv')
contrasts=get(SOURCE/'Figure5_source_data_contrasts.csv')
rob=get(SOURCE/'Figure5_source_data_robustness.csv')
cached={}
for name in ['07B_SEVERITY_DEFINITION_ROBUSTNESS.csv','07_FIG5_PANEL_C_CONDITIONAL_QUANTILE_TRENDS.csv','07F_AR6_CONSENSUS.csv','07_FIG5_PANEL_F_ROBUSTNESS_MATRIX.csv']:
    cached[name]=get(TABLES/name)
def frozen_csv(path):return cached[Path(path).name].copy()
old_dec=get(TABLES/'07A_SEVERITY_DECILE_TP_TRENDS.csv')
cmp=deciles.merge(old_dec,on=['product','severity_decile'],suffixes=('_final','_old'))
assert np.allclose(cmp.slope,cmp.estimate)
for c in ['CI_low','CI_high']:assert np.allclose(cmp[c+'_final'],cmp[c+'_old'])
ref=cached['07B_SEVERITY_DEFINITION_ROBUSTNESS.csv']
cmp=interaction.merge(ref,on=['product','model'],suffixes=('_final','_old'))
assert np.allclose(cmp.beta,cmp.beta_year_x_severity)
for c in ['CI_low','CI_high']:assert np.allclose(cmp[c+'_final'],cmp[c+'_old'])
pd.testing.assert_frame_equal(rob,cached['07_FIG5_PANEL_F_ROBUSTNESS_MATRIX.csv'])
cond=get(ROOT/'result1.0/diagnostics/figure5_diagnostics/Figure5_source_data_conditional_quantiles.csv')
cmp=cond.merge(cached['07_FIG5_PANEL_C_CONDITIONAL_QUANTILE_TRENDS.csv'],left_on=['product','severity_decile','quantile'],right_on=['product','severity_decile','TP_quantile'])
assert np.allclose(cmp.slope,cmp.sen_slope_months_decade)
assert set(cond.n_years)=={63}
g=interaction[interaction['product'].eq('GRUN')&interaction.model.eq('main')].iloc[0]
assert g.CI_low<0<g.CI_high
print('Final/Part7 values verified; main GRUN interaction CI crosses zero.')


In [ ]:

TABLES=PART7_ROOT/'tables'
import logging
log=logging.getLogger('figtotal.Figure5')
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import regionmask
PRODUCTS = ['GRUN', 'ERA5', 'GLDAS']
PCOLOR = {'GRUN':'#007C78','ERA5':'#CB7935','GLDAS':'#667BA5'}
MODEL_LABEL = {'main':'Cumulative', 'duration':'Duration controlled', 'peak':'Peak severity'}
MODEL_MARKER = {'main':'o', 'duration':'s', 'peak':'D'}
QLEVELS = [.10, .25, .50, .75, .90]
QCOLS = ['Q10','Q25','Q50','Q75','Q90']

plt.rcParams.update({
    'font.family':'Arial', 'font.size':7.4, 'axes.titlesize':8.4,
    'axes.labelsize':7.5, 'xtick.labelsize':6.8, 'ytick.labelsize':6.8,
    'legend.fontsize':6.7, 'axes.linewidth':.65, 'xtick.major.width':.6,
    'ytick.major.width':.6, 'xtick.major.size':3, 'ytick.major.size':3,
    'axes.spines.top':False, 'axes.spines.right':False,
    'pdf.fonttype':42, 'ps.fonttype':42, 'svg.fonttype':'none',
    'figure.facecolor':'white', 'axes.facecolor':'white'
})


ar6_frame=gpd.read_file('zip://'+str(REPO/'assets/geography/IPCC-WGI-reference-regions-v4.zip'))
AR6_LAND=regionmask.from_geopandas(ar6_frame.loc[ar6_frame.Type.str.contains('Land')], names='Name', abbrevs='Acronym', name='AR6 reference regions (land only)', source='Iturbide et al., 2020 (ESSD)', overlap=False)
def read(name):
    path = TABLES/name
    if not path.exists(): raise FileNotFoundError(path)
    d = frozen_csv(path); log.info('READ %s shape=%s', path, d.shape); return d
def style_axis(ax):
    ax.tick_params(direction='out',pad=2);ax.grid(False)
def panel_label(ax,label,x=-.13,y=1.08):
    ax.text(x,y,label,transform=ax.transAxes,ha='left',va='top',fontsize=10.5,fontweight='bold',clip_on=False)
def plot_b(ax):
    d=read('07B_SEVERITY_DEFINITION_ROBUSTNESS.csv');models=['main','duration','peak']
    ybase=np.arange(3)[::-1];offsets={'main':.20,'duration':0,'peak':-.20}
    for i,p in enumerate(PRODUCTS):
        for m in models:
            r=d[(d['product']==p)&(d.model==m)].iloc[0];y=ybase[i]+offsets[m]
            lo,est,hi=float(r.CI_low),float(r.beta_year_x_severity),float(r.CI_high)
            xx=np.linspace(lo,hi,100);left=max(est-lo,1e-8);right=max(hi-est,1e-8)
            zz=np.where(xx<=est,(xx-est)/left,(xx-est)/right)
            envelope=.075*np.sqrt(np.clip(1-zz**2,0,1))
            ax.fill_between(xx,y,y+envelope,color=PCOLOR[p],alpha=.20,lw=0,zorder=1)
            ax.hlines(y,lo,hi,color=PCOLOR[p],lw=1.05,zorder=2)
            ax.scatter(est,y,marker=MODEL_MARKER[m],s=20,color=PCOLOR[p],edgecolor='white',linewidth=.35,zorder=3)
    ax.axvline(0,color='#999999',lw=.65,ls=(0,(3,2)),zorder=0)
    ax.set(yticks=ybase,yticklabels=PRODUCTS,xlabel=r'β(year × severity) (months decade⁻¹)',title='Year × severity: definition sensitivity')
    ax.set_ylim(-.48,2.52);ax.set_xlim(-.25,.04)
    handles=[Line2D([],[],marker=MODEL_MARKER[m],ls='',mfc='#666666',mec='white',label=MODEL_LABEL[m],ms=5) for m in models]
    ax.legend(handles=handles,frameon=False,loc='lower left',ncol=1,handletextpad=.35,labelspacing=.22)
    ax.text(.985,.035,'Envelope spans 95% bootstrap CI',transform=ax.transAxes,ha='right',va='bottom',fontsize=5.8,color='#666666')
    style_axis(ax);panel_label(ax,'b')
def plot_c(fig,spec):
    d=read('07_FIG5_PANEL_C_CONDITIONAL_QUANTILE_TRENDS.csv');lim=np.nanmax(np.abs(d.sen_slope_months_decade));norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim);cmap='RdBu_r';axs=[]
    sub=spec.subgridspec(3,1,hspace=.20)
    for i,p in enumerate(PRODUCTS):
        ax=fig.add_subplot(sub[i]);g=d[d['product']==p].pivot(index='severity_decile',columns='TP_quantile_label',values='sen_slope_months_decade').reindex(columns=QCOLS)
        im=ax.imshow(g,origin='lower',aspect='auto',cmap=cmap,norm=norm,extent=(-.5,4.5,.5,10.5),interpolation='nearest')
        ax.set_yticks([1,5,10]);ax.set_ylabel(p,rotation=0,labelpad=15,va='center',fontweight='bold',fontsize=6.9)
        if i<2:ax.tick_params(labelbottom=False)
        else:ax.set(xticks=range(5),xticklabels=QCOLS,xlabel='Propagation-time quantile')
        for s in ax.spines.values():s.set_visible(False)
        axs.append(ax)
    axs[0].set_title('Severity-conditioned quantile trends',pad=5)
    axs[1].text(-.20,.5,'Severity decile',rotation=90,transform=axs[1].transAxes,ha='center',va='center',fontsize=7.4)
    cb=fig.colorbar(im,ax=axs,orientation='vertical',fraction=.05,pad=.025);cb.set_label('Sen slope\n(months decade⁻¹)',fontsize=6.8);cb.ax.tick_params(labelsize=6)
    panel_label(axs[0],'c',x=-.16,y=1.22);return axs
def plot_e(fig,spec):
    ax=fig.add_subplot(spec,projection=ccrs.Robinson());d=read('07F_AR6_CONSENSUS.csv').set_index('AR6');regions=AR6_LAND
    lim=np.nanmax(np.abs(d.median_beta));norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim);cmap=plt.get_cmap('RdBu_r')
    for name,poly in zip(regions.abbrevs,regions.polygons):
        value=d.loc[name,'median_beta'] if name in d.index else np.nan
        ax.add_geometries([poly],ccrs.PlateCarree(),facecolor=cmap(norm(value)) if np.isfinite(value) else '#F1F1F1',edgecolor='#777777',linewidth=.28)
        if name in d.index and d.loc[name,'n_negative']>=2:
            pt=poly.representative_point();solid=d.loc[name,'n_negative']==3
            ax.plot(pt.x,pt.y,'o',ms=3.0,mfc='#222222' if solid else 'white',mec='#222222',mew=.55,transform=ccrs.PlateCarree(),zorder=5)
    ax.set_extent([-180,180,-60,85],ccrs.PlateCarree());ax.set_title('Regional consistency across AR6 regions',pad=4)
    cax=inset_axes(ax,width='82%',height='4.2%',loc='lower center',bbox_to_anchor=(0,-.16,1,1),bbox_transform=ax.transAxes,borderpad=0)
    cb=fig.colorbar(plt.cm.ScalarMappable(norm=norm,cmap=cmap),cax=cax,orientation='horizontal');cb.set_label(r'Median β(year × severity)',fontsize=6.8,labelpad=2);cb.ax.tick_params(labelsize=6,pad=1)
    handles=[Line2D([],[],marker='o',ls='',mfc='#222222',mec='#222222',ms=4,label='3/3 negative'),Line2D([],[],marker='o',ls='',mfc='white',mec='#222222',ms=4,label='2/3 negative')]
    ax.legend(handles=handles,loc='lower left',bbox_to_anchor=(.02,-.025),frameon=False,ncol=2,fontsize=6)
    stats=f"{int((d.median_beta<0).sum())}/{len(d)} median β < 0\n{int((d.n_negative==3).sum())} regions: 3/3 negative\n{int((d.n_negative>=2).sum())} regions: ≥2/3 negative"
    ax.text(.985,.05,stats,transform=ax.transAxes,ha='right',va='bottom',fontsize=6.1,bbox=dict(fc='white',ec='none',alpha=.78,pad=1.5))
    panel_label(ax,'e',x=-.075,y=1.08);return ax
def plot_f(fig,spec):
    ax=fig.add_subplot(spec);d=robustness_table();cols=['Main','Duration','Peak','Common grid','Min 20','Min 30','Min 50']
    ybase={'GRUN':2,'ERA5':1,'GLDAS':0};offsets=dict(zip(cols,np.linspace(.12,-.12,len(cols))))
    group_marker={'Main':'o','Duration':'o','Peak':'o','Common grid':'s','Min 20':'^','Min 30':'^','Min 50':'^'}
    for p in PRODUCTS:
        g=d[d['product']==p].set_index('setting').loc[cols].reset_index();yb=ybase[p]
        ax.hlines(yb,g.beta.min(),g.beta.max(),color=PCOLOR[p],alpha=.22,lw=2.2,zorder=1)
        for _,r in g.iterrows():
            y=yb+offsets[r.setting]
            if r.beta>=0:
                ax.scatter(r.beta,y,marker='x',s=24,color=PCOLOR[p],linewidth=.9,zorder=4)
            else:
                filled=r.CI_high<0
                ax.scatter(r.beta,y,marker=group_marker[r.setting],s=24,facecolor=PCOLOR[p] if filled else 'white',
                           edgecolor=PCOLOR[p],linewidth=.8,zorder=4)
        ax.scatter(g.beta.median(),yb,marker='D',s=43,facecolor='#222222',edgecolor='white',linewidth=.55,zorder=5)
    ax.axvline(0,color='#999999',lw=.65,ls=(0,(3,2)),zorder=0)
    ax.set(yticks=[2,1,0],yticklabels=PRODUCTS,xlabel=r'β(year × severity) (months decade⁻¹)',title='Cross-product robustness')
    ax.set_ylim(-.48,2.48);ax.set_xlim(-.25,.035)
    shape_handles=[Line2D([],[],marker='o',ls='',mfc='#777777',mec='#777777',ms=4,label='Definitions'),
                   Line2D([],[],marker='s',ls='',mfc='#777777',mec='#777777',ms=4,label='Common grid'),
                   Line2D([],[],marker='^',ls='',mfc='#777777',mec='#777777',ms=4,label='Min. events'),
                   Line2D([],[],marker='D',ls='',mfc='#222222',mec='white',ms=5,label='Median')]
    ax.legend(handles=shape_handles,frameon=False,loc='lower left',ncol=2,handletextpad=.25,columnspacing=.7,labelspacing=.25)
    ax.text(.985,.035,'Filled: CI < 0   Open: CI overlaps 0',transform=ax.transAxes,ha='right',va='bottom',fontsize=5.7,color='#666666')
    style_axis(ax);panel_label(ax,'f')
def check_canvas(fig):
    fig.canvas.draw();ren=fig.canvas.get_renderer();bad=[]
    for t in fig.findobj(matplotlib.text.Text):
        if not t.get_visible() or not t.get_text():continue
        b=t.get_window_extent(ren);fb=fig.bbox
        if b.width and b.height and (b.x0 < -1 or b.y0 < -1 or b.x1 > fb.width+1 or b.y1 > fb.height+1):bad.append(t.get_text())
    return bad

def robustness_table():
    return frozen_csv(TABLES/'07_FIG5_PANEL_F_ROBUSTNESS_MATRIX.csv')




In [ ]:
DRAW_DIR=ROOT/'new_fig_try/fig5/calculated_results/panel_d'
draws=pd.read_csv(DRAW_DIR/'paired_bootstrap_draws.csv')
assert len(draws)==4500
def plot_a(ax):
    for p in PRODUCTS:
        d=deciles[deciles['product'].eq(p)].sort_values('severity_decile')
        ax.plot(d.severity_decile,d.slope,'o-',color=PCOLOR[p],lw=1.8 if p=='GRUN' else 1.2,ms=3,label=p)
        ax.fill_between(d.severity_decile,d.CI_low,d.CI_high,color=PCOLOR[p],alpha=.10,lw=0)
    ax.axhline(0,color='.6',lw=.6,ls='--')
    ax.set(xlim=(.7,10.3),xticks=[1,3,5,7,10],xlabel='Fixed within-grid severity decile',
           ylabel='PT Sen slope (months decade⁻¹)',title='Severity-dependent retrospective trends')
    ax.legend(frameon=False,loc='upper right',fontsize=7)
    ax.text(.03,.03,'Shading: existing 95% intervals',transform=ax.transAxes,fontsize=6)
    style_axis(ax);panel_label(ax,'a')

def plot_d(ax):
    offsets=[-.23,0,.23]
    markers=['v','o','^']
    for i,p in enumerate(PRODUCTS):
        for j,q in enumerate([.1,.5,.9]):
            samples=draws[draws['product'].eq(p)&np.isclose(draws['quantile'],q)].contrast.to_numpy()
            pos=i+offsets[j]
            vp=ax.violinplot([samples],positions=[pos],widths=.20,showextrema=False,points=120,bw_method='scott')
            for body in vp['bodies']:
                body.set_facecolor(PCOLOR[p]);body.set_edgecolor(PCOLOR[p]);body.set_alpha(.24);body.set_linewidth(.65)
            ref=contrasts[contrasts['product'].eq(p)&np.isclose(contrasts['quantile'],q)].iloc[0]
            ax.vlines(pos,ref.CI_low,ref.CI_high,color=PCOLOR[p],lw=1.4,zorder=3)
            ax.scatter(pos,ref.high_minus_low_slope,marker=markers[j],s=20,color=PCOLOR[p],edgecolor='white',linewidth=.35,zorder=4)
    ax.axhline(0,color='.6',ls='--',lw=.6)
    ax.set(xticks=range(3),xticklabels=PRODUCTS,xlim=(-.55,2.55),
      ylabel='High − low severity trend contrast (months decade⁻¹)',
      title='High–low severity bootstrap distributions')
    handles=[Line2D([],[],marker=m,ls='',color='.4',label=q,ms=4) for m,q in zip(markers,['Q10','Q50','Q90'])]
    ax.legend(handles=handles,loc='lower left',frameon=False,ncol=3,fontsize=6,handletextpad=.25,columnspacing=.6)
    style_axis(ax);panel_label(ax,'d')
fig=plt.figure(figsize=(7.20472440945,180/25.4),facecolor='white')
outer=fig.add_gridspec(2,3,left=.075,right=.925,bottom=.15,top=.91,wspace=.49,hspace=.57)
axa=fig.add_subplot(outer[0,0]);plot_a(axa)
axb=fig.add_subplot(outer[0,1]);plot_b(axb)
plot_c(fig,outer[0,2])
axd=fig.add_subplot(outer[1,0]);plot_d(axd)
plot_e(fig,outer[1,1])
plot_f(fig,outer[1,2])
fig.text(.07,.03,'1952–2014. Fixed within-grid severity ranks; retrospective timing associations only. Magnitude and uncertainty depend on product and specification.\nRegional map: existing three-product median; markers denote negative estimates, not significance. No causal or prospective-warning claim.',fontsize=7,linespacing=1.5)


In [ ]:

FIGURE_NUMBER=5
FROZEN_TITLES={'a': 'Retrospective trends vary with severity', 'b': 'Severity associations depend on definition', 'c': 'Quantile trends vary with severity', 'd': 'High–low contrasts show bootstrap uncertainty', 'e': 'Regional severity associations vary geographically', 'f': 'Interaction estimates vary across specifications'}
from matplotlib.text import Text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import textwrap
OUTDIR=Path(OUT if FIGURE_NUMBER==6 else OUTPUT_ROOT)
QA=OUTDIR/'qa';QA.mkdir(exist_ok=True)
def numerical_artists():
    result={}
    def add(key,value):
        a=np.ma.asarray(value)
        if a.dtype.kind in 'biufc':
            v=np.ascontiguousarray(a.filled(np.nan) if np.ma.isMaskedArray(a) and a.dtype.kind in 'fc' else a)
            result[key]={'shape':list(v.shape),'sha256':hashlib.sha256(v.tobytes()).hexdigest()}
    for j,a in enumerate(fig.axes):
        for k,l in enumerate(a.lines):add(f'{j}/line{k}/x',l.get_xdata());add(f'{j}/line{k}/y',l.get_ydata())
        for k,l in enumerate(a.collections):
            if l.get_array() is not None:add(f'{j}/collection{k}/array',l.get_array())
            if hasattr(l,'get_offsets'):add(f'{j}/collection{k}/offsets',l.get_offsets())
            if hasattr(l,'get_segments'):
                for h,v in enumerate(l.get_segments()):add(f'{j}/collection{k}/segment{h}',v)
            if l.__class__.__name__ in ('PolyCollection','FillBetweenPolyCollection'):
                for h,v in enumerate(l.get_paths()):add(f'{j}/collection{k}/path{h}',v.vertices)
        for k,l in enumerate(a.images):add(f'{j}/image{k}',l.get_array())
        for k,l in enumerate(a.patches):
            if isinstance(l,matplotlib.patches.Rectangle):add(f'{j}/bar{k}',[l.get_x(),l.get_y(),l.get_width(),l.get_height()])
    return result
fig.canvas.draw()
before=numerical_artists()
contracts=json.loads((REPO/'assets/other/panel_contracts.json').read_text(encoding='utf-8'))
oldtitles={r[2] for r in contracts if r[0]==FIGURE_NUMBER}
notes=[]
for t in list(fig.findobj(Text)):
    s=t.get_text()
    if s in oldtitles or s in list('abcdef'):
        t.set_text('');continue
    if len(s)>180:
        notes.append(s);t.set_text('');continue
    s=s.replace('decade$^{-1}$','decade⁻¹').replace(r'$\beta$','β').replace('$R^2$','R²')
    t.set_text(s);t.set_fontfamily('DejaVu Sans');t.set_fontsize(5.5 if FIGURE_NUMBER in (1,2) else 6)
    if t.get_rotation():t.set_rotation_mode('anchor')
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial','Liberation Sans'],
    'font.size':6,'axes.labelsize':6,'xtick.labelsize':5.5,'ytick.labelsize':5.5,
    'legend.fontsize':5.5,'pdf.fonttype':42,'svg.fonttype':'none'})
DIV=plt.get_cmap('RdBu_r')
for ax in fig.axes:
    cb=getattr(ax,'_colorbar',None)
    for sp in ax.spines.values():sp.set_linewidth(.45);sp.set_edgecolor('#657076')
    ax.tick_params(width=.45,length=2,pad=2,labelsize=5.5)
    ax.xaxis.label.set_fontsize(6);ax.yaxis.label.set_fontsize(6)
    ax.title.set_fontsize(6)
    if cb is not None:
        cb.outline.set_linewidth(.35)
        ax.tick_params(length=1.5,pad=1)
        ax.set_axes_locator(None)
        p=ax.get_position()
        if cb.orientation=='horizontal':ax.set_position([p.x0,p.y0,p.width,min(p.height,2.5/(72*fig.get_figheight()))])
        else:ax.set_position([p.x0,p.y0,min(p.width,2.5/(72*fig.get_figwidth())),p.height])
    for artist in list(ax.images)+list(ax.collections):
        norm=getattr(artist,'norm',None)
        if norm is not None and norm.vmin is not None and norm.vmax is not None and norm.vmin<0<norm.vmax:
            assert np.isclose(abs(norm.vmin),abs(norm.vmax)), 'Original diverging limits require review'
            artist.set_cmap(DIV);artist.set_norm(TwoSlopeNorm(vmin=norm.vmin,vcenter=0,vmax=norm.vmax))
    for line in ax.lines:
        if line.get_linestyle() in ('--',':'):
            x=np.asarray(line.get_xdata());y=np.asarray(line.get_ydata())
            if (len(x)>0 and np.all(x==0)) or (len(y)>0 and np.all(y==0)):
                line.set_color('#8B9298');line.set_linewidth(.5);line.set_linestyle('--')
        if line.get_markersize()>4:line.set_markersize(3.5)
        if line.get_linewidth()>1.8:line.set_linewidth(1.5)
    if hasattr(ax,'projection'):
        for col in ax.collections:
            if col.__class__.__name__=='FeatureArtist':col.set_linewidth(.25)
        ax.spines['geo'].set_linewidth(.45)

if FIGURE_NUMBER==1:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='24 mo':t.set_x(.62);t.set_transform(ax.transAxes);t.set_y(.07);t.set_ha('right')
            if t.get_text().startswith('d1  '):t.set_position((0,1.02));t.set_va('bottom')
            if t.get_text().startswith('Common 36-month'):
                notes.append(t.get_text());t.set_text('Common 36-month\ncoverage cohort');t.set_position((.04,.06));t.set_fontsize(5)
if FIGURE_NUMBER==2:
    for ax in fig.axes:
        if ax.get_xlabel()=='Months relative to meteorological anchor':ax.set_xlabel('Months from meteorological anchor',fontsize=6)
        for t in ax.texts:
            if t.get_text().startswith('Negative ='):notes.append(t.get_text());t.set_text('')
            if '95% block CI' in t.get_text():t.set_position((.03,1.02));t.set_va('bottom');t.set_fontsize(5)
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb is not None and 'hexagon' in ax.get_xlabel():
            from matplotlib.ticker import FixedLocator,FixedFormatter,NullLocator
            ax.xaxis.set_major_locator(FixedLocator([1,10,100,1000]));ax.xaxis.set_major_formatter(FixedFormatter(['1','10','100','1000']));ax.xaxis.set_minor_locator(NullLocator())
if FIGURE_NUMBER==3:
    for leg in list(fig.legends):
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            handles=leg.legend_handles;labels=[t.get_text() for t in leg.get_texts()]
            pos=gs[0,0].get_position(fig);leg.remove()
            fig.legend(handles,labels,ncol=4,frameon=False,fontsize=5.2,loc='upper left',bbox_to_anchor=(pos.x0-.005,pos.y0-.038),handlelength=1.2,columnspacing=.8)
    maps=[a for a in fig.axes if hasattr(a,'projection')]
    for a in maps:
        a.title.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Main widths pool events'):
            t.set_text('Main widths: pooled events. d2/f: grid-year means. Grey: no estimate.');t.set_fontsize(5.5)
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Thin: annual'):
                t.set_position((.02,1.01));t.set_va('bottom');t.set_text('Thin: annual; thick: 5-year visual mean');t.set_fontsize(5)
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='horizontal':
            p=ax.get_position();ax.set_position([p.x0,p.y0-.01,p.width,p.height]);cb.set_ticks([cb.norm.vmin,0,cb.norm.vmax] if cb.norm.vmin<0 else [0,50,100])
        if ax.images:
            p=ax.get_position();ax.set_position([p.x0+.04,p.y0,p.width-.04,p.height]);ax.tick_params(axis='y',labelsize=5)
if FIGURE_NUMBER==4:
    for leg in fig.legends:leg.set_bbox_to_anchor((.5,.045))
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text().replace('Parent number','Antecedent count').replace('Many-to-one','Multiple antecedents')
            if '  ρ: ' in s:
                s=s.replace('  ρ: ','\nρ: ');t.set_position((.01,1.03));t.set_va('bottom');t.set_fontsize(5.2)
            if s.startswith('a1  '):
                t.set_text('');p=ax.get_position();fig.text(p.x0,p.y1+.013,'a1  Event scale · GRUN',fontsize=5.5,fontweight='bold')
            else:t.set_text(s)
            if not ax.axison:t.set_fontsize(5.2)
        if ax.get_ylabel()=='TP slope':ax.set_ylabel('PT slope',fontsize=6)
        cb=getattr(ax,'_colorbar',None)
        if cb is not None:cb.set_ticks([-1,0,1])
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text()
            if s.startswith('Envelope spans') or s.startswith('Filled:'):
                t.set_position((0,-.22));t.set_ha('left');t.set_va('top');t.set_fontsize(5.2)
            if s=='Severity decile':t.set_x(-.29);t.set_fontsize(5.5)
            if 'median β < 0' in s:
                t.set_position((.0,-.32));t.set_ha('left');t.set_va('top');t.set_text(s.replace('\n','; ',1));t.set_fontsize(5.2);t.set_bbox(None)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._loc=2
            for cax in fig.axes:
                cb=getattr(cax,'_colorbar',None)
                if cb and cb.orientation=='horizontal':
                    p=ax.get_position();cax.set_position([p.x0,p.y0-.105,p.width,2.5/(72*fig.get_figheight())])
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='threshold floor':t.set_text('AUC < 0.60: L* = 0');t.set_x(.34);t.set_fontsize(5.5)
            if t.get_text().startswith('Δ ='):
                t.set_x(1.0);t.set_fontsize(5);t.set_bbox(None)
                if ax.get_title()=='Retrospective PT':
                    t.set_x(.02);t.set_ha('left');t.set_text(t.get_text().replace('Δ = ','Δ '))
            if t.get_text() in ('GRUN','ERA5','GLDAS') and ax in fig.axes[12:]:
                t.set_position((.50,.84));t.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Points: temporal'):t.set_y(.025);t.set_fontsize(5)


for ax in fig.axes:
    leg=ax.get_legend()
    if leg:
        for t in leg.get_texts():t.set_fontsize(5)
    for t in ax.texts:
        if len(t.get_text())>36:t.set_text(textwrap.fill(t.get_text(),36,break_long_words=False));t.set_fontsize(5)
    if len(ax.get_xlabel())>32:ax.set_xlabel(textwrap.fill(ax.get_xlabel(),32,break_long_words=False),fontsize=5.5)
if FIGURE_NUMBER==3:
    for t in fig.texts:
        if t.get_text().startswith(('f1 ','f2 ','Bands:')):t.set_text(textwrap.fill(t.get_text(),34,break_long_words=False));t.set_fontsize(5)
    for leg in fig.legends:
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            h=leg.legend_handles;l=[t.get_text() for t in leg.get_texts()];p=gs[0,0].get_position(fig);leg.remove();fig.legend(h,l,ncol=2,frameon=False,fontsize=5,loc='upper left',bbox_to_anchor=(p.x0,p.y0-.04))
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='vertical':
            p=ax.get_position();ax.set_position([.94,p.y0,p.width,p.height]);ax.yaxis.label.set_fontsize(5)
        for t in ax.texts:
            if t.get_text()=='Severity decile':t.set_x(-.40)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._ncols=1
            for t in ax.texts:
                if 'median β' in t.get_text():t.set_y(-.45)
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        if ax.get_xlabel().startswith('Sen slope'):ax.set_xlabel('Sen slope\n(months decade⁻¹)',fontsize=5)
        cb=getattr(ax,'_colorbar',None)
        if cb:
            p=ax.get_position();ax.set_position([p.x0,p.y0-.016,p.width,p.height])
        if ax.get_title()=='Retrospective PT':ax.set_xticks([-.08,0,.04])
    for leg in fig.legends:leg.set_bbox_to_anchor((bottom[0].get_position(fig).x0,.078))
    for t in fig.texts:
        if t.get_text().startswith('Δ = high'):t.set_y(.046);t.set_text('Δ = high − low; bars:\ncomponent 95% CIs');t.set_fontsize(5)
        if t.get_text().startswith('Points: temporal'):t.set_text('Points: temporal Sen slopes;\nbars: 95% Sen intervals');t.set_y(.035);t.set_fontsize(5)


if FIGURE_NUMBER==3:
    for ax in fig.axes:
        if ax.get_title().startswith(('f1 ','f2 ')):ax.set_title(textwrap.fill(ax.get_title(),29),loc='left',fontsize=5,pad=8)
    for t in fig.findobj(Text):
        if t.get_text().startswith(('f1 ','f2 ')):t.set_text(textwrap.fill(t.get_text(),29));t.set_fontsize(5)
        if t.get_text().startswith('Bands:'):t.set_y(t.get_position()[1]-.027)
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='vertical':
            p=ax.get_position();ax.set_position([.923,p.y0,p.width,p.height]);ax.yaxis.labelpad=1
        leg=ax.get_legend()
        if leg and any(t.get_text() in ('Q50','Min. events') for t in leg.get_texts()):leg.set_bbox_to_anchor((0,-.18));leg._loc=2
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Filled:'):t.set_y(-.34)
if FIGURE_NUMBER==6:
    for leg in fig.legends:leg.set_bbox_to_anchor((bottom[0].get_position(fig).x0,.095))

if FIGURE_NUMBER in (1,2,3):specs=[gs[0,0],gs[0,1],gs[0,2],gs[1,0],gs[1,1],gs[1,2]]
elif FIGURE_NUMBER==4:specs=[gs[0,0],gs[0,1],gs[1,0],gs[1,1]]
elif FIGURE_NUMBER==5:specs=[outer[0,0],outer[0,1],outer[0,2],outer[1,0],outer[1,1],outer[1,2]]
else:specs=top+bottom
headers=[]
for letter,spec in zip('abcdef',specs):
    b=spec.get_position(fig);h=fig.get_figheight()*72;w=fig.get_figwidth()*72
    width_chars=33 if FIGURE_NUMBER in (1,2) else 28
    title_offset=9 if FIGURE_NUMBER in (1,2) else 24 if FIGURE_NUMBER==4 else 18
    txt=textwrap.fill(FROZEN_TITLES[letter],width_chars,break_long_words=False)
    t=fig.text(b.x0,b.y1+title_offset/h,txt,ha='left',va='bottom',fontsize=6.5,fontweight='normal',linespacing=1.4,color='#20272B')
    fig.text(b.x0-10/w,b.y1+title_offset/h,letter,ha='left',va='bottom',fontsize=8,fontweight='bold',color='#20272B')
    headers.append(t)
fig.canvas.draw()
after=numerical_artists()
assert before==after,'A scientific artist changed during styling'
(QA/'numerical_artist_invariance.json').write_text(json.dumps({'status':'PASS','entries':len(before),'before':before,'after':after},indent=2),encoding='utf-8')
(QA/'retained_caption_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
from audit_panel_alignment import require_matplotlib_panel_alignment
eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None and not hasattr(a,'projection')]
if len(eligible)<2:eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None]
ids=[f'plot{i+1}' for i in range(len(eligible))]
rows={};cols={}
for ident,ax in zip(ids,eligible):
    p=ax.get_position();rows.setdefault((round(p.y0,5),round(p.y1,5)),[]).append(ident);cols.setdefault((round(p.x0,5),round(p.x1,5)),[]).append(ident)
rg=[v for v in rows.values() if len(v)>1];cg=[v for v in cols.values() if len(v)>1]
ex=[]
for group in cg:
    if len(group)>=3:
        parent_ids={str(eligible[ids.index(k)].get_subplotspec().get_gridspec()) for k in group}
        ex.append({'panels':group,'checks':['vertical-gutter'],'reason':'Column spans distinct main-panel rows and nested stacks; outer and inner gutters intentionally differ. Stack spacing remains uniform within each original nested GridSpec.'})
for group in rg:
    if len(group)>=3:
        widths=[eligible[ids.index(k)].get_position().width for k in group]
        if np.ptp(widths)>1e-4:
            ex.append({'panels':group,'checks':['panel-width','horizontal-gutter'],'reason':'Nested composites intentionally allocate widths to distinct quantities; within-product repeats are compared separately.'})
require_matplotlib_panel_alignment(fig,axes=eligible,panel_ids=ids,row_groups=rg,column_groups=cg,
    exemptions=ex,require_panel_labels=False,json_out=QA/'alignment.json',overlay_svg=QA/'alignment.svg',strict=True)
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.png',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.pdf',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.svg',dpi=450,facecolor='white')
fig.canvas.draw()
meta=[{'axis':j,'bounds':list(a.get_position().bounds),'title':a.get_title(),'xlabel':a.get_xlabel(),'ylabel':a.get_ylabel(),
       'texts':[t.get_text() for t in a.texts]} for j,a in enumerate(fig.axes)]
(QA/'axes.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Figure {FIGURE_NUMBER}: exported, {len(before)} numerical artist arrays unchanged.')
plt.close(fig)
